# 01 · data exploration

the spotify million playlist dataset (MPD) ships as 1000 JSON slice files, 1000 playlists each. for development i load the first 20 slices (20k playlists, ~1.3M interactions, ~260k unique tracks) so the loop from raw json to a trained model fits in a few minutes.


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, matplotlib.pyplot as plt


In [ ]:
from src.data import load_dataset
ds = load_dataset(source='mpd', cache_dir=ROOT/'data/processed',
                  slice_dir=str(ROOT/'data/raw'), rebuild=False)
print(f'playlists: {ds.n_playlists():,}')
print(f'unique tracks: {ds.n_tracks():,}')
print(f'interactions: {len(ds.interactions):,}')


## playlist length distribution


In [ ]:
pl_len = ds.interactions.groupby('playlist_id').size()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(pl_len, bins=50, color='#8b3a3a', alpha=0.85)
ax.set_xlabel('tracks per playlist'); ax.set_ylabel('playlists')
ax.set_title('playlist length distribution'); plt.show()


## track popularity (long tail as expected)


In [ ]:
track_pop = ds.interactions.groupby('track_id').size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(track_pop.values[:5000], color='#1f1c1a')
ax.set_xlabel('track rank'); ax.set_ylabel('# playlists containing track')
ax.set_yscale('log'); ax.set_title('track popularity (top 5000)')
plt.show()


the top-1% of tracks dominate the playlist appearances by orders of magnitude. classical CF will lean on that signal; the hybrid model should help most on tracks in the mid-tail where collaborative evidence is sparse.
